In [1]:
import torch 
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset,DataLoader
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import torch.optim as optim

In [2]:
# extra tokenization stuffs
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\junio\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\junio\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
document = """The Last Train Journey

On a cold winter evening, Daniel arrived at the old railway station carrying a small leather bag.
The station was nearly empty except for a few passengers waiting quietly on wooden benches.
A loud whistle echoed through the air as the final train of the night slowly entered the platform.

Daniel checked the ticket in his pocket and walked toward the train.
The metal doors opened with a creaking sound and warm air rushed outside.
Inside the train, yellow lights flickered softly above rows of empty seats.

He chose a seat beside the window and placed his bag carefully near his feet.
As the train began to move, the city lights slowly disappeared behind thick clouds of fog.
Snow started falling gently outside while distant mountains appeared under the moonlight.

Across the aisle sat an elderly woman reading an old book with a blue cover.
Every few minutes she looked outside the window as if searching for something familiar.
Daniel noticed a silver necklace around her neck shaped like a small compass.

After several hours, the train stopped at a remote station surrounded by tall pine trees.
Very few people entered or left the train at that place.
A strange silence covered the station as the doors closed once again.

Curious about the woman, Daniel finally started a conversation.
The woman introduced herself as Eleanor and explained that she was returning to her hometown after many years.
She said the town was hidden deep within the northern mountains and was known for its beautiful frozen lake.

As the journey continued, Eleanor shared stories from her childhood.
She described colorful festivals, music performances, and lanterns floating across the lake during winter nights.
Daniel listened carefully while the train moved through snowy valleys and dark forests.

Near midnight, the train suddenly slowed down because heavy snow had covered the tracks ahead.
Passengers became nervous and began discussing possible delays.
Conductors moved through the cabins trying to calm everyone.

Daniel looked outside and saw workers removing snow under bright floodlights.
The freezing wind shook the train slightly while snow continued falling from the dark sky.
Despite the delay, Eleanor remained calm and continued telling her stories.

Several hours later, the tracks were finally cleared and the train resumed its journey.
The passengers sighed with relief as warm coffee was served throughout the cabins.
Daniel and Eleanor continued talking about travel, history, and forgotten places around the world.

Just before sunrise, the train reached the mountain town.
The station was small but beautifully decorated with glowing lamps and snow-covered rooftops.
People wearing thick winter coats walked slowly across the icy streets.

Before leaving, Eleanor handed Daniel a folded piece of paper.
Inside it was a drawing of the frozen lake along with a short message thanking him for the conversation.
Daniel smiled and carefully placed the paper inside his notebook.

As the train departed once again, Daniel stood on the platform watching the sunrise over the mountains.
The cold air, quiet streets, and distant church bells created a peaceful atmosphere he would never forget.
Years later, Daniel would still remember that mysterious winter journey and the stories shared by Eleanor on the last train of the night.
"""

##### Tokenize

In [5]:
tokens = word_tokenize(document.lower())

##### Building Vocabulary

In [12]:
Counter(tokens).keys()

dict_keys(['the', 'last', 'train', 'journey', 'on', 'a', 'cold', 'winter', 'evening', ',', 'daniel', 'arrived', 'at', 'old', 'railway', 'station', 'carrying', 'small', 'leather', 'bag', '.', 'was', 'nearly', 'empty', 'except', 'for', 'few', 'passengers', 'waiting', 'quietly', 'wooden', 'benches', 'loud', 'whistle', 'echoed', 'through', 'air', 'as', 'final', 'of', 'night', 'slowly', 'entered', 'platform', 'checked', 'ticket', 'in', 'his', 'pocket', 'and', 'walked', 'toward', 'metal', 'doors', 'opened', 'with', 'creaking', 'sound', 'warm', 'rushed', 'outside', 'inside', 'yellow', 'lights', 'flickered', 'softly', 'above', 'rows', 'seats', 'he', 'chose', 'seat', 'beside', 'window', 'placed', 'carefully', 'near', 'feet', 'began', 'to', 'move', 'city', 'disappeared', 'behind', 'thick', 'clouds', 'fog', 'snow', 'started', 'falling', 'gently', 'while', 'distant', 'mountains', 'appeared', 'under', 'moonlight', 'across', 'aisle', 'sat', 'an', 'elderly', 'woman', 'reading', 'book', 'blue', 'cover

In [14]:
vocab = {'<unk>':0}

In [15]:
for token in Counter(tokens).keys():
    if token not in vocab:
        vocab[token] = len(vocab)
    

In [17]:
len(vocab)

284

##### Extract sentence from data

In [19]:
input_sentences = document.split('\n')

In [33]:
def text_to_indices(sentence,vocab):
    numerical_sentence = []
    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])
        else:
            numerical_sentence.append(vocab['<unk>'])
    return numerical_sentence

In [34]:
input_numerical_sentences= []
for sentence in input_sentences:
    input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()),vocab))

In [35]:
input_numerical_sentences

[[1, 2, 3, 4],
 [],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18, 19, 20, 21],
 [1, 16, 22, 23, 24, 25, 26, 6, 27, 28, 29, 30, 5, 31, 32, 21],
 [6, 33, 34, 35, 36, 1, 37, 38, 1, 39, 3, 40, 1, 41, 42, 43, 1, 44, 21],
 [],
 [11, 45, 1, 46, 47, 48, 49, 50, 51, 52, 1, 3, 21],
 [1, 53, 54, 55, 56, 6, 57, 58, 50, 59, 37, 60, 61, 21],
 [62, 1, 3, 10, 63, 64, 65, 66, 67, 68, 40, 24, 69, 21],
 [],
 [70, 71, 6, 72, 73, 1, 74, 50, 75, 48, 20, 76, 77, 48, 78, 21],
 [38, 1, 3, 79, 80, 81, 10, 1, 82, 64, 42, 83, 84, 85, 86, 40, 87, 21],
 [88, 89, 90, 91, 61, 92, 93, 94, 95, 96, 1, 97, 21],
 [],
 [98, 1, 99, 100, 101, 102, 103, 104, 101, 14, 105, 56, 6, 106, 107, 21],
 [108, 27, 109, 110, 111, 61, 1, 74, 38, 112, 113, 26, 114, 115, 21],
 [11, 116, 6, 117, 118, 119, 120, 121, 122, 123, 6, 18, 124, 21],
 [],
 [125, 126, 127, 10, 1, 3, 128, 13, 6, 129, 16, 130, 131, 132, 133, 134, 21],
 [135, 27, 136, 43, 137, 138, 1, 3, 13, 139, 140, 21],
 [6, 141, 142, 143, 1, 16, 38, 1, 54, 144, 145, 146

##### Sequence forming

In [38]:
training_sequence = []
for sentence in input_numerical_sentences:
    for i in range(1,len(sentence)):
        training_sequence.append(sentence[:i+1])

In [39]:
training_sequence

[[1, 2],
 [1, 2, 3],
 [1, 2, 3, 4],
 [5, 6],
 [5, 6, 7],
 [5, 6, 7, 8],
 [5, 6, 7, 8, 9],
 [5, 6, 7, 8, 9, 10],
 [5, 6, 7, 8, 9, 10, 11],
 [5, 6, 7, 8, 9, 10, 11, 12],
 [5, 6, 7, 8, 9, 10, 11, 12, 13],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18, 19],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18, 19, 20],
 [5, 6, 7, 8, 9, 10, 11, 12, 13, 1, 14, 15, 16, 17, 6, 18, 19, 20, 21],
 [1, 16],
 [1, 16, 22],
 [1, 16, 22, 23],
 [1, 16, 22, 23, 24],
 [1, 16, 22, 23, 24, 25],
 [1, 16, 22, 23, 24, 25, 26],
 [1, 16, 22, 23, 24, 25, 26, 6],
 [1, 16, 22, 23, 24, 25, 26, 6, 27],
 [1, 16, 22, 23, 24, 25, 26, 6, 27, 28],
 [1, 16, 22, 23, 24, 2

In [45]:
max_len = 0
for outside in training_sequence:
    if len(outside) > max_len:
        max_len = len(outside)

In [46]:
max_len

25

In [47]:
padded_training_sequence = []
for sequence in training_sequence:
    padded_training_sequence.append([0]*(25 - len(sequence))+sequence)

In [49]:
padded_training_sequence[0]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2]

##### Converting to tensors

In [50]:
padded_training_sequence = torch.tensor(padded_training_sequence,dtype=torch.long)

In [51]:
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        ...,
        [  0,   0, 158,  ...,   3,  40,   1],
        [  0, 158, 223,  ...,  40,   1,  41],
        [158, 223,  10,  ...,   1,  41,  21]])

In [54]:
X = padded_training_sequence[:,:-1]
y = padded_training_sequence[:,-1]

In [59]:
len(X),len(y)

(555, 555)

##### Custom dataset & dataloader

In [60]:
class CustomDataset(Dataset):
    def __init__(self,X,y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self,idx):
        return self.X[idx],self.y[idx]

In [61]:
dataset = CustomDataset(X,y)

In [62]:
dataloader = DataLoader(dataset=dataset,batch_size=32,shuffle=True)

In [65]:
dataloader

In [68]:
X.shape,y.shape

(torch.Size([555, 24]), torch.Size([555]))

In [66]:
for input,output in dataloader:
    print(input)
    print(output)

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   6,  33,  34,  35,  36,   1,  37,  38],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  38,   1,   3, 268,
         145, 146,  10,  11, 269,   5,   1,  44, 270,   1],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0, 108],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   1,  53],
        [  0,   0,   0,   0,   0,   0,   0,   0,  62, 259,  22,   6, 260,  40,
           1, 168, 169, 261,  56,   6, 262, 263, 264, 265],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,  11, 184,  76,  92,   1,   3],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   5,   6,
           7,   8,   9,  10,  11,  12,  13,   1,  14,  15],
        [  0,   0,   0,   0